# automl-competition-kit — benchmark on Colab

Runs the benchmark where a GPU is free. The image engine is the reason to be
here: measuring it on a laptop CPU produces a number about the hardware, not
about the code.

**Runtime → Change runtime type → T4 GPU** before you start.

Order: 1 GPU check → 2 get the repo → 3 install → 4 Kaggle token →
5 verify → 6 image benchmark → 7 download the results.

## 1. Hardware

In [ ]:
!nvidia-smi || echo "NO GPU — set Runtime > Change runtime type > T4 GPU"

## 2. Get the repo

Fill in your GitHub URL. If the repo is still private, use the upload cell
underneath instead.

In [ ]:
REPO_URL = "https://github.com/<your-user>/automl-competition-kit.git"

import os
if not os.path.exists("automl-competition-kit"):
    !git clone $REPO_URL
%cd automl-competition-kit
!ls

In [ ]:
# ALTERNATIVE to the cell above: upload a zip of the repo instead of cloning.
# Skip this cell if the clone worked.
#
# from google.colab import files
# up = files.upload()
# name = list(up)[0]
# !unzip -q -o $name
# %cd automl-kit

## 3. Install

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -r requirements-image.txt
!pip install -q -e .
!pip install -q kaggle

## 4. Kaggle token

Upload the `kaggle.json` you already have on your machine
(`%USERPROFILE%\\.kaggle\\kaggle.json`). It stays in this Colab session only.

In [ ]:
import os, json
from google.colab import files

os.makedirs("/root/.kaggle", exist_ok=True)
if not os.path.exists("/root/.kaggle/kaggle.json"):
    up = files.upload()                       # pick kaggle.json
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(up["kaggle.json"])
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("username:", json.load(open("/root/.kaggle/kaggle.json"))["username"])

## 5. Environment check, slug check, leakage demo

In [ ]:
!python benchmark/run_benchmark.py --check
!python benchmark/run_benchmark.py --verify
!python benchmark/leakage_demo.py

## 6. Image benchmark — the point of running here

`--per-class` caps how many images per class are used, which is what keeps the
run bounded. 300 per class over 6 classes is roughly 25 minutes on a T4 for the
fast preset plus the baseline; the full preset is capped by `--full-budget`.

Start with `--skip-full` to get numbers quickly, then re-run without it.

In [ ]:
!python benchmark/image_benchmark.py --per-class 300 --skip-full

In [ ]:
# Once the fast numbers look right, the full preset as well (45+ minutes):
# !python benchmark/image_benchmark.py --per-class 300

In [ ]:
# A different dataset — any Kaggle dataset laid out as one folder per class:
# !python benchmark/image_benchmark.py --slug tongpython/cat-and-dog --per-class 400

## 7. Tabular benchmark (optional here)

Colab's free tier gives 2 vCPUs, so the boosted-tree datasets are usually
*faster on a laptop*. Run this here only if you would rather not tie up your
own machine.

In [ ]:
# !python benchmark/run_benchmark.py --skip-full

## 8. Take the results with you

In [ ]:
from google.colab import files
files.download("benchmark/results.json")